In [ ]:
import ROOT
import uuid
%jsroot on

file = ROOT.TFile.Open("zhh_AIDA.root", "READ")
tree_nmc = file.Get("KinFitQQ_NMC")
tree_solvenu = file.Get("KinFitQQ_solveNu") 

canvas_name = f"canvas_{uuid.uuid4().hex[:6]}"
canvas = ROOT.TCanvas(canvas_name, "ZHH Analysis", 1500, 800)
canvas.Divide(4, 2)

ROOT.gStyle.SetOptStat(0)

# Explicit configuration for every single pad
plot_configs = [
    {"type": "standard", "var": "FitErrorCode", "logy": False, "title": "Fit Error Code"},
    {"type": "standard", "var": "FitProbability", "logy": True, "title": "Fit Probability"},
    {"type": "standard", "var": "FitChi2", "logy": True, "title": "Fit Chi2"},
    {"type": "standard", "var": "pullJetEnergy", "logy": False, "title": "Jet Energy Pull"},
    {"type": "mass", "before": "ZMassBeforeFit", "after": "ZMassAfterFit", "title": "Z Mass"},
    {"type": "mass", "before": "HHMassBeforeFit", "after": "HHMassAfterFit", "title": "HH Mass"},
    {"type": "mass", "before": "ZHHMassBeforeFit", "after": "ZHHMassAfterFit", "title": "ZHH Mass"}
]

for i, config in enumerate(plot_configs):
    canvas.cd(i + 1)
    
    # Check if this pad needs a log scale
    if config.get("logy"):
        ROOT.gPad.SetLogy(1)
    else:
        ROOT.gPad.SetLogy(0)
        
    if config["type"] == "mass":
        # --- MASS VARIABLES (3 Lines) ---
        var_before = config["before"]
        var_after = config["after"]
        title = config["title"]
        
        # Raw Data (Black)
        tree_nmc.SetLineColor(ROOT.kBlack)
        tree_nmc.Draw(f"{var_before} >> raw_{i}", "")
        raw = ROOT.gDirectory.Get(f"raw_{i}")
        if raw: raw.SetTitle(f"{title}; Mass (GeV); Events")
        
        # NMC (Red)
        tree_nmc.SetLineColor(ROOT.kRed)
        tree_nmc.Draw(f"{var_after} >> nmc_{i}", "", "SAME")
        nmc = ROOT.gDirectory.Get(f"nmc_{i}")
        
        # Solvenu (Blue)
        tree_solvenu.SetLineColor(ROOT.kBlue)
        tree_solvenu.Draw(f"{var_after} >> solvenu_{i}", "", "SAME")
        solvenu = ROOT.gDirectory.Get(f"solvenu_{i}")
        
        # Overshoot protection
        if raw and nmc and solvenu:
            raw.SetMaximum(max(raw.GetMaximum(), nmc.GetMaximum(), solvenu.GetMaximum()) * 1.2)
            
        leg_cmd = f"leg_{i} = ROOT.TLegend(0.25, 0.70, 0.50, 0.90)\n"
        leg_cmd += f"leg_{i}.SetBorderSize(0)\n"
        leg_cmd += f"leg_{i}.AddEntry(raw, 'No fit', 'l')\n"
        leg_cmd += f"leg_{i}.AddEntry(nmc, 'NMC', 'l')\n"
        leg_cmd += f"leg_{i}.AddEntry(solvenu, 'Solvenu', 'l')\n"
        leg_cmd += f"leg_{i}.Draw()"
        exec(leg_cmd)
        
    elif config["type"] == "standard":
        # --- STANDARD VARIABLES (2 Lines) ---
        var = config["var"]
        title = config["title"]
        
        # NMC (Red)
        tree_nmc.SetLineColor(ROOT.kRed)
        tree_nmc.Draw(f"{var} >> nmc_{i}", "")
        nmc = ROOT.gDirectory.Get(f"nmc_{i}")
        if nmc: nmc.SetTitle(f"{title}; Value; Events")
        
        # Solvenu (Blue)
        tree_solvenu.SetLineColor(ROOT.kBlue)
        tree_solvenu.Draw(f"{var} >> solvenu_{i}", "", "SAME")
        solvenu = ROOT.gDirectory.Get(f"solvenu_{i}")
        
        # Overshoot protection
        if nmc and solvenu:
            nmc.SetMaximum(max(nmc.GetMaximum(), solvenu.GetMaximum()) * 1.2)
            
        leg_cmd = f"leg_{i} = ROOT.TLegend(0.70, 0.75, 0.90, 0.90)\n"
        leg_cmd += f"leg_{i}.SetBorderSize(0)\n"
        leg_cmd += f"leg_{i}.AddEntry(nmc, 'NMC', 'l')\n"
        leg_cmd += f"leg_{i}.AddEntry(solvenu, 'Solvenu', 'l')\n"
        leg_cmd += f"leg_{i}.Draw()"
        exec(leg_cmd)

canvas.Draw()